# 14강 실습: 표준국어대사전 XML 파일 처리

14·15강은 임건수(2025)를 다시 해 보는 두 강의다. 논문은 표준국어대사전에서 형용사 후보 목록('어근+하다'를 뺀 2,631개 → 동모음 탈락을 뺀 1,008개)을 만들어 메신저 말뭉치에서 어미 없이 쓰인 어간('부럽', '귀엽')을 찾았다. 오늘은 그 목록의 재료인 사전의 XML 파일 88개를 **표제어 표 하나**로 펼쳐 CSV로 남긴다.

> 임건수(2025), 매체 언어에서 형용사 어간의 자립적 사용 양상 연구, 《한국어학》 107, 183–208. https://doi.org/10.20405/kl.2025.05.107.183

**오늘 할 것**

1. XML 글을 Python의 요소(`Element`)로 읽고, 경로를 따라 원하는 요소를 꺼낸다.
2. 실제 사전 파일 하나에서 '부럽다'를 찾아 읽는다.
3. 88개 파일을 모두 읽어 표제어 표를 만들고 `data/stdict/headwords.csv`로 저장한다.

**시작하기 전에**

- 이 노트북은 저장소 **맨 위**(`/workspaces/LDS2026`)에 두어야 `data/stdict/…` 경로가 맞는다.
- eTL에서 받은 `stdict_xml_20261012.zip`을 `data/stdict/`에 올리고 터미널에서 풀어 둔다.

```bash
cd /workspaces/LDS2026/data/stdict
unzip -q stdict_xml_20261012.zip -d xml
```

- 사전은 2026년 10월 12일에 받은 버전이다(표제어 436,587개). 이용 조건은 크리에이티브 커먼즈 저작자표시-동일조건변경허락 2.0 대한민국이고, 출전이 있는 용례는 원저작권자의 허락이 필요하다. zip과 XML, CSV는 `data/` 아래라 커밋되지 않는다.
- 커밋하기 전에는 [Restart] 후 [Run All]로 위에서 아래까지 한 번에 돌려 본다.

## 0. 준비

네 가지를 불러온다.

- **xml.etree.ElementTree**: XML을 읽고 쓰는 Python의 표준 라이브러리. `json`처럼 설치하지 않아도 된다. 이름이 길어 `ET`로 줄여 부른다. 외부 패키지 `lxml`도 같은 이름의 함수(`fromstring`, `find`, `findall`, `iter`)를 쓴다.
- **glob**: 패턴에 맞는 파일 이름을 모은다(11강).
- **pandas**: 7강에서 쓴 표 라이브러리.
- **tqdm**: 반복이 얼마나 진행되었는지 막대로 보여 준다. 10강에서 `kiwipiepy`를 설치할 때 함께 설치되었다.

In [1]:
import xml.etree.ElementTree as ET
from glob import glob
import pandas as pd
from tqdm import tqdm

## 1. XML을 Python으로

**요소(element)** 는 시작 태그 `<pos>`와 끝 태그 `</pos>` 사이에 내용을 둔 것이다. 시작 태그 안에 `이름="값"`으로 적은 것은 **속성(attribute)** 이다. 사전은 품사 번호를 `<pos_code>148758001</pos_code>`처럼 자식 요소로 적었고 속성은 하나도 쓰지 않았지만, 여기서는 속성으로 적어 본다.

- `ET.fromstring(s)`: XML 글을 `Element`로 바꾼다. `json.loads`가 글을 `dict`로 바꾸던 것과 같은 자리다.
- `.tag`: 요소의 이름, `.text`: 요소의 내용(글), `.get('code')`: 속성 `code`의 값

In [2]:
s = '<pos code="148758001">형용사</pos>'
e = ET.fromstring(s)
print(type(s))
print(type(e))
print(e.tag, e.text, e.get('code'))

<class 'str'>
<class 'xml.etree.ElementTree.Element'>
pos 형용사 148758001


'부럽다'를 몇 겹 걷어 내고 간추린 표제어다. 요소 안에 요소가 들어간다. 안에 들어간 요소를 **자식**, 감싼 요소를 **부모**라고 부른다. 가장 바깥의 `item`이 **뿌리**이고, 문서 하나에 뿌리는 꼭 하나다.

XML에는 배열이 없다. 같은 이름의 자식(`conjugation` 둘)을 차례대로 늘어놓으면 그것이 목록이다.

- `len(item)`: **자식**의 수. 손자 요소는 세지 않는다.
- `for child in item`: 자식이 차례로 나온다.

In [3]:
x = '''<item><target_code>148758</target_code>
 <word_info><word>부럽다</word><word_type>고유어</word_type>
  <conjugation>부러워</conjugation><conjugation>부러우니</conjugation>
  <pos_info><pos>형용사</pos></pos_info></word_info></item>'''
item = ET.fromstring(x)
print(item.tag, len(item))
print([child.tag for child in item])

item 2
['target_code', 'word_info']


- `find('word_info')`: 그 이름의 **첫** 자식 요소. 빗금(`/`)으로 이은 경로를 따라 내려가기도 한다.
- `findtext('target_code')`: 찾은 요소의 `.text`까지 한 번에.

경로는 11강의 `c['utterance'][1]['form']`과 달리 이름으로만 내려간다. XML은 내용의 꼴을 적지 않으므로 번호도 **문자열**로 나온다. 수로 쓰려면 `int()`로 바꾼다.

In [4]:
info = item.find('word_info')
print(info.find('word').text)
print(item.find('word_info/pos_info/pos').text)
print(item.findtext('target_code'))
print(type(item.findtext('target_code')))

부럽다
형용사
148758
<class 'str'>


- `findall`: 그 이름의 자식을 **모두** 목록으로 돌려준다. 같은 이름을 되풀이한 목록을 읽는 함수다.
- `iter`: 깊이와 상관없이 그 이름의 요소를 **모두** 찾는다.

`item.findall('conjugation')`이 0인 까닭: `conjugation`은 `item`의 자식이 아니라 손자다. `find`와 `findall`은 자식만, 경로를 주면 그 경로만 본다.

In [5]:
for c in info.findall('conjugation'):
    print(c.text)
print(len(item.findall('conjugation')))
print(len(list(item.iter('conjugation'))))

부러워
부러우니
0
2


이 표제어에는 `origin`(어원)이 없다. `find`는 못 찾으면 `None`을 돌려준다.

`info.find('origin').text`는 `AttributeError: 'NoneType' object has no attribute 'text'`를 내고 멈춘다(이 줄은 노트북에 넣지 않았다). `None`에는 `.text`가 없기 때문이다. `findtext`는 멈추지 않고 `None`을 돌려준다. 없을 수도 있는 요소에는 이쪽을 쓴다.

**문법이 틀리면.** XML의 약속(대소문자를 가린다, 연 요소는 연 순서의 반대로 닫는다, 글 속의 `&`는 `&amp;`로 적는다, 뿌리는 하나다)을 어기면 `ET.fromstring`이 `ParseError`를 낸다(노트북에 넣지 않았다).

- `ET.fromstring('<pos>형용사</Pos>')` → `ParseError: mismatched tag: line 1, column 10`
- `ET.fromstring('<word>R&B</word>')` → `ParseError: not well-formed (invalid token): line 1, column 9`

칸은 0부터 센다. 메시지는 틀린 곳이 아니라 **알아챈 곳**을 알려 준다.

In [6]:
print(info.find('origin'))
print(info.findtext('origin'))

None
None


**요소를 다시 글로.** `ET.Element`로 요소를 새로 만들고 내용을 넣은 뒤 `ET.tostring`으로 적는다.

- `encoding='unicode'`: 바이트가 아니라 `str`로 달라는 뜻이다.

글 속의 꺾쇠와 `&`는 저절로 `&lt;`, `&gt;`, `&amp;`가 되고, 다시 읽으면 원래 글로 돌아온다. XML을 손으로 이어 붙여 만들지 말고 이렇게 만드는 까닭이다. 사전은 꺾쇠를 적는 다른 방법인 **CDATA**(`<![CDATA[ … ]]>`, 안쪽을 풀지 않고 글자 그대로 읽는다)로 표제어와 뜻풀이를 감쌌다.

In [7]:
d = ET.Element('example')
d.text = '<부럽다> & <귀엽다>'
print(ET.tostring(d, encoding='unicode'))
print(ET.fromstring(ET.tostring(d, encoding='unicode')).text)

<example>&lt;부럽다&gt; &amp; &lt;귀엽다&gt;</example>
<부럽다> & <귀엽다>


## 2. 사전 파일 하나

'부럽다'가 든 파일을 연다.

- `ET.parse(path)`: 파일을 읽는다. 인코딩은 첫 줄의 선언(`<?xml version="1.0" encoding="UTF-8"?>`)을 따른다.
- `getroot()`: 뿌리 요소를 준다.

뿌리 `channel` 아래에 설명 다섯과 표제어 `item` 5,000개가 나란히 있다. `total`은 사전 전체의 표제어 수이고 88개 파일에 모두 같다.

In [8]:
path = 'data/stdict/xml/1610292_165000.xml'
root = ET.parse(path).getroot()
print(root.tag)
print([child.tag for child in root][:7])
print(root.findtext('total'), len(root.findall('item')))

channel
['title', 'link', 'description', 'lastBuildDate', 'total', 'item', 'item']
436587 5000


표제어 5,000개를 차례로 보며 표제어(`word`)가 '부럽다'인 것만 남긴다(11강의 목록 만들기). 하나다. 이제 `item`은 앞 절의 간추린 표제어가 아니라 **진짜** '부럽다'다. 활용형은 `conju_info`, `conjugation_info`, `conjugation` 세 겹 안에 있고, 뜻풀이는 `pos_info`, `comm_pattern_info`, `sense_info` 아래에 있다.

In [9]:
found = [it for it in root.findall('item')
         if it.findtext('word_info/word') == '부럽다']
print(len(found))
item = found[0]
print(item.findtext('target_code'))
print([child.tag for child in item.find('word_info')])

1
148758
['word', 'word_unit', 'word_type', 'pronunciation_info', 'conju_info', 'conju_info', 'origin', 'pos_info']


- `repr`: 보이지 않는 글자까지 이스케이프로 보여 준다. `\n`은 줄바꿈, `\t`는 탭이다.

`item`의 내용은 `<item>` 바로 뒤의 줄바꿈과 탭 둘이다. JSON과 달리 XML에서는 태그 사이의 공백도 **글**로 남는다. 그래서 자식을 담은 요소의 `.text`는 쓰지 않는다. 한편 `word`의 `<![CDATA[…]]>`는 사라지고 글만 남았다.

In [10]:
print(repr(item.text))
print(repr(item.findtext('word_info/word')))

'\n\t\t'
'부럽다'


세 겹 아래의 활용형은 `iter`가 깊이와 상관없이 모아 온다. 뜻풀이는 경로를 끝까지 적었다. 용례는 8개이고, 그 가운데 셋은 소설에서 따온 것이라 `source`(출전)가 붙어 있다. 이용 조건이 따로 말한 용례다.

In [11]:
info = item.find('word_info')
print([c.text for c in info.iter('conjugation')])
print(info.findtext('pos_info/comm_pattern_info/sense_info/definition'))
print(len(list(info.iter('example'))))

['부러워', '부러우니']
남의 좋은 일이나 물건을 보고 자기도 그런 일을 이루거나 그런 물건을 가졌으면 하고 바라는 마음이 있다.
8


**CDATA 안의 글자 참조.** 다른 파일에서 '괄호'의 뜻풀이를 꺼낸다.

- `.//`: 경로 앞에 붙이면 '어느 깊이든'이다. 처음 만나는 `definition`을 찾는다.

중괄호 자리에 `&#123; &#125;`가 남았다. `&#123;`는 코드 포인트 123번 글자, 곧 `{`다(9강). 사전이 `{ }`를 글자 참조로 바꾼 뒤 CDATA로 감싸서, CDATA 안이라 **풀리지 않았다**. 이런 뜻풀이가 152개다. 뜻풀이를 쓸 때는 `html.unescape`로 풀 수 있다.

In [12]:
root2 = ET.parse('data/stdict/xml/1610292_40000.xml').getroot()
for it in root2.findall('item'):
    if it.findtext('word_info/word') == '괄호':
        print(it.findtext('.//definition'))

문장 부호의 하나. 소괄호(( )), 중괄호(&#123; &#125;), 대괄호([ ])가 있다.


## 3. 사전 전체를 표로

파일 이름을 모은다. `sorted`는 문자열을 **글자 하나씩** 비교한다. `10000.xml`과 `100000.xml`은 여섯째 글자에서 `.`(46)이 `0`(48)보다 앞서고, `5000`은 한참 뒤로 간다. 사전의 가나다순과 어긋난다.

In [13]:
paths = sorted(glob('data/stdict/xml/*.xml'))
print(len(paths))
for p in paths[:3]:
    print(p)

88
data/stdict/xml/1610292_10000.xml
data/stdict/xml/1610292_100000.xml
data/stdict/xml/1610292_105000.xml


**수의 순서로 줄 세우기.**

- `number(path)`: 파일 이름 끝의 수를 `int`로 돌려준다. `'1610292_10000.xml'` → `'10000.xml'` → `'10000'` → `10000`
- `key=number`: 이름 대신 `number(이름)`을 비교하라는 뜻이다.

이제 표제어가 사전의 순서대로 나온다. 앞의 `1610292`가 달라도 그대로 쓸 수 있다.

In [14]:
def number(path):
    return int(path.split('_')[-1].removesuffix('.xml'))

paths = sorted(paths, key=number)
for p in paths[:3]:
    print(p)

data/stdict/xml/1610292_5000.xml
data/stdict/xml/1610292_10000.xml
data/stdict/xml/1610292_15000.xml


**88개 파일을 차례로.** 파일마다, 표제어마다, **품사마다** `dict` 하나를 담는다. 번호는 `int()`로 수로 바꾼다. 어종(`word_type`)은 없는 표제어가 있어서 `findtext`로 꺼낸다.

- `tqdm(paths)`: `paths`를 그대로 하나씩 내주면서, 몇 개째인지와 남은 시간을 막대로 그린다. 오래 걸리는 반복에 씌운다.

In [15]:
rows = []
for path in tqdm(paths):
    for item in ET.parse(path).getroot().findall('item'):
        code = int(item.findtext('target_code'))
        info = item.find('word_info')
        for pos in info.findall('pos_info/pos'):
            rows.append({'target_code': code,
                         'word': info.findtext('word'),
                         'word_unit': info.findtext('word_unit'),
                         'word_type': info.findtext('word_type'),
                         'pos': pos.text})

100%|██████████| 88/88 [00:14<00:00,  6.22it/s]


표제어는 436,587개인데 행은 439,566개다. 첫 행이 사전의 첫 표제어 'ㄱ'이다. 표제어의 `-`는 사전의 **붙임표**로, 구성 요소의 경계를 보인다.

- `word_unit`: 단어, 구, 속담, 관용구
- `word_type`: 고유어, 한자어, 외래어, 혼종어

In [16]:
headwords = pd.DataFrame(rows)
print(headwords.shape)
print(headwords.head(3))

(439566, 5)
   target_code    word word_unit word_type pos
0          443       ㄱ        단어       고유어  명사
1          445   ㄱㄴㄷ-순        단어       혼종어  명사
2       386697  ㄱㄴㄷ-차례        단어       혼종어  명사


**한 행은 '표제어 하나의 품사 하나'다.** '바삭하다'는 형용사(보송보송하다)이자 동사(바삭 소리가 나다)라서 `pos_info`가 둘이고, 표에서 두 행이 된다. 이렇게 2,979행이 늘었다. 8강의 물음 '무엇을 하나로 셀 것인가'다. 표제어를 세려면 `nunique`를 쓴다.

In [17]:
print(headwords['target_code'].nunique())
print(headwords[headwords['word'] == '바삭-하다'])

436587
        target_code   word word_unit word_type  pos
139773       428825  바삭-하다        단어       고유어   동사
139774       428825  바삭-하다        단어       고유어  형용사


**비어 있는 칸.** 속담과 관용구에는 `<word_type>`이 없다. `findtext`의 `None`이 표에서 `NaN`이 되었다. `find('word_type').text`로 적었다면 첫 속담에서 멈췄을 것이다.

- `to_dict()`: 결과를 `dict`로 바꾸어 한 줄로 찍는다(12강).

In [18]:
print(headwords.isna().sum().to_dict())
na = headwords['word_type'].isna()
print(headwords.loc[na, 'word_unit'].value_counts().to_dict())

{'target_code': 0, 'word': 0, 'word_unit': 0, 'word_type': 11322, 'pos': 0}
{'속담': 7436, '관용구': 3886}


**품사별로 세어 보면.** '품사 없음'은 대부분 구(63,004행)다.

- `to_string()`: 맨 아래의 `Name`, `dtype` 줄을 빼고 찍는다.

논문의 첫 단계를 흉내 낸다. 형용사 13,400행에서 '-하다'로 끝나는 7,747행을 빼면 5,653행, 논문의 2,631개의 **두 배가 넘는다**. 까닭은 15강에서 찾는다.

In [19]:
counts = headwords['pos'].value_counts()
print(counts.head(6).to_string())
adj = headwords[headwords['pos'] == '형용사']
print(len(adj))
print(adj['word'].str.endswith('-하다').sum())

pos
명사       270046
품사 없음     70774
동사        56474
형용사       13400
부사        12325
구         11322
13400
7747


## 4. 저장하기

저장하고 다시 읽어 본다. 빈칸은 어종이 없는 11,322칸 그대로이고, 번호는 미리 `int()`로 바꾸어 두었으니 수로 돌아온다. 파일은 19MB다. 657MB의 XML에서 다섯 열만 남겼다. 15강은 XML이 아니라 이 CSV에서 시작한다.

In [20]:
headwords.to_csv('data/stdict/headwords.csv', index=False)
back = pd.read_csv('data/stdict/headwords.csv')
print(back.shape)
print(back.isna().sum().sum())
print(back['target_code'].dtype)

(439566, 5)
11322
int64


## 이 표가 말하지 않는 것

- **버전**: 2026년 10월 12일에 받은 사전이다. 2025년 이 강의에서 받은 버전은 436,145개였다. 받은 날이 다르면 수가 달라진다.
- **뜻**: 다섯 열만 남겼다. 뜻풀이, 활용형, 용례는 표에 없다. 필요하면 XML로 돌아가 같은 방법으로 꺼낸다.
- **한 행**: 표제어 하나가 아니라 표제어 하나의 품사 하나다.

## 다음 시간

15강(11월 4일 수요일)은 `headwords.csv`에서 시작한다. 논문의 후보 목록(형용사 2,631개, 1,008개)을 다시 만들어 보고, 오늘의 5,653이 2,631과 왜 다른지 살핀다. 표제어 `창연-하다02`가 첫 실마리다.

논문은 어미 없이 쓰인 어간의 65%가 'ㅂ' 받침이라고 했다. 그런데 사전의 형용사에서 'ㅂ' 받침 어간은 **원래** 몇 %일까? 그 수를 모르면 65%가 많은지 알 수 없다.